# 2. The Container Stacking Rules Problem

## Tier 1 — Mathematical formulation (made runnable via exhaustive search)

Container stacking rules determine where arriving containers are placed in a yard bay. A bad stacking decision creates **blocking**: when an early-departing container is buried under later-departing containers, you must do extra moves (**reshuffles**).

### Learning goals

- Understand how **departure times** create a natural “retrieval order”.
- Learn a simple and concrete definition of **blocking** and **reshuffles**.
- See how an “optimal” placement can be found for a **small** instance.

### What this notebook outputs

- An optimal stacking plan for a small example instance.
- The number of reshuffles implied by that plan.
- A visualization of stacks and blocking relationships.

### Why we do exhaustive search here

The Tier 1 text describes an integer programming formulation. To keep this notebook fully runnable without external solvers, we compute an optimal solution for a small instance by **enumerating all feasible placements**.

This gives the same learning outcome ("what does optimal mean?") while keeping dependencies simple.

In [1]:
# Environment check (no installs here)
#
# Best practice: dependencies should be preinstalled in the JupyterHub/Docker image.
# If you're running locally, install packages once in your Python environment.

from typing import List, Tuple, Optional, Dict

try:
    import itertools
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

print("Dependencies imported successfully.")

## Concrete example instance (from the scenario)

We will use the small example with **5 containers** and known departure times:

- C1: departure time 3
- C2: departure time 1 (earliest)
- C3: departure time 5
- C4: departure time 2
- C5: departure time 6

And a yard bay with:

- 3 stacks (Stack 1..3)
- 2 tiers per stack (bottom tier 0, top tier 1)

### Key rule: what counts as a reshuffle?

When retrieving containers in increasing departure time order, a container causes reshuffles if it is **blocked** by containers above it.

A simple way to count reshuffles in a fixed stack configuration:

- Look at each stack from bottom → top.
- Every pair where a **bottom** container departs earlier than a container **above** it is a blocking relationship.
- Each blocking relationship contributes 1 reshuffle.

In [2]:
# ----------------------------
# Example data
# ----------------------------
# We model each container with:
# - an id (C1..C5)
# - a departure time d (smaller means it must be retrieved earlier)
#
# Important idea:
# If a container with early departure time is below a container with later departure time,
# then we will need to reshuffle the top container to access the bottom one.

containers = [
    ("C1", 3),
    ("C2", 1),
    ("C3", 5),
    ("C4", 2),
    ("C5", 6),
]

# Yard bay shape for this Tier 1 example
NUM_STACKS = 3
MAX_TIERS = 2

# Quick lookup: container_id -> departure_time
dep_time = {cid: d for cid, d in containers}

containers

In [ ]:
# ----------------------------
# Counting blocking relationships / reshuffles (simple model)
# ----------------------------
# We need a clear, beginner-friendly definition that we can compute.
#
# In one stack, containers are stored bottom -> top.
# Retrieval order is increasing departure time (earlier departure retrieved first).
#
# A blocking relationship exists when:
# - a container A is below container B
# - A must depart earlier than B
#
# In that case, when we want to retrieve A, we must temporarily move B away.
# We count that as 1 reshuffle.


def count_blocking_pairs_in_stack(stack_bottom_to_top: List[str]) -> int:
    """Count blocking pairs (reshuffles) inside ONE stack."""

    reshuffles = 0

    # Compare every lower container with every container above it.
    for lower_idx in range(len(stack_bottom_to_top)):
        for upper_idx in range(lower_idx + 1, len(stack_bottom_to_top)):
            lower = stack_bottom_to_top[lower_idx]
            upper = stack_bottom_to_top[upper_idx]

            # If lower departs earlier, then upper blocks lower.
            if dep_time[lower] < dep_time[upper]:
                reshuffles += 1

    return reshuffles


def count_total_reshuffles(stack_config: List[List[str]]) -> int:
    """Count total reshuffles across all stacks."""

    return sum(count_blocking_pairs_in_stack(stack) for stack in stack_config)


# Quick sanity check on a tiny artificial stack:
# - Bottom departs at time 1
# - Top departs at time 5
# That should create 1 blocking pair.
_test_stack = ["C2", "C3"]
count_blocking_pairs_in_stack(_test_stack)

## Step 1 — Exhaustive search for an optimal stacking plan (small instance)

A real integer programming (IP) model typically needs a solver.

To keep this notebook runnable with basic Python only, we do the following for this small instance:

- Assume containers arrive in the given order: C1, C2, C3, C4, C5.
- For each arriving container, we choose a stack (1..3).
- The container is always placed on **top** of that stack.
- If a stack is full (already has 2 tiers), that choice is infeasible.

We enumerate all feasible stack-choice sequences and pick the one with the smallest number of reshuffles.

In [ ]:
# ----------------------------
# Enumerate feasible placements
# ----------------------------
# We'll represent a yard configuration as a list of stacks.
# Each stack is a list of container IDs, from bottom -> top.


def apply_choice_sequence(choice_seq: Tuple[int, ...]) -> Optional[List[List[str]]]:
    """Build a stack configuration from a sequence of stack choices.

    Parameters
    - choice_seq: a tuple of length 5, each element in {0,1,2}
      (0 means Stack 1, 1 means Stack 2, 2 means Stack 3)

    Returns
    - stack_config if feasible
    - None if any stack exceeds MAX_TIERS
    """

    # Start with empty stacks
    stack_config: List[List[str]] = [[] for _ in range(NUM_STACKS)]

    # Containers arrive in the given order
    arrival_order = [cid for cid, _ in containers]

    for cid, stack_idx in zip(arrival_order, choice_seq):
        # If stack is full, this sequence is infeasible
        if len(stack_config[stack_idx]) >= MAX_TIERS:
            return None

        # Otherwise, place the container on top
        stack_config[stack_idx].append(cid)

    return stack_config


# Enumerate all possible stack-choice sequences
# 3 choices per container, 5 containers => 3^5 = 243 sequences (small)
all_choice_sequences = list(itertools.product(range(NUM_STACKS), repeat=len(containers)))
len(all_choice_sequences)

In [ ]:
# ----------------------------
# Evaluate all feasible sequences and pick the best
# ----------------------------
# We'll keep track of:
# - the best (minimum) reshuffle count
# - the corresponding stack configuration
# - one example sequence that achieves it

best_reshuffles = None
best_config = None
best_seq = None

feasible_count = 0

for seq in all_choice_sequences:
    config = apply_choice_sequence(seq)
    if config is None:
        continue

    feasible_count += 1

    # Compute reshuffles with our blocking-pairs metric
    resh = count_total_reshuffles(config)

    if best_reshuffles is None or resh < best_reshuffles:
        best_reshuffles = resh
        best_config = config
        best_seq = seq

feasible_count, best_reshuffles, best_seq, best_config

## Step 2 — Inspect the best solution (make it human-readable)

Optimization code is only useful if we can **interpret** the solution.

In terminal yard operations, we typically want to see:

- the final stack configuration (which container is in which stack/tier)
- which containers block which (the root cause of reshuffles)
- a clear visualization that operators and students can understand

Next, we convert the best configuration into a table and generate these visual outputs.

In [ ]:
# ----------------------------
# Convert the best configuration into a table
# ----------------------------
# We want a clear mapping:
# container -> stack -> tier -> departure time

assert best_config is not None

rows = []
for s_idx, stack in enumerate(best_config, start=1):
    for tier_idx, cid in enumerate(stack):
        rows.append(
            {
                "container": cid,
                "stack": s_idx,
                "tier": tier_idx,  # 0=bottom
                "departure_time": dep_time[cid],
            }
        )

sol_df = pd.DataFrame(rows).sort_values(["stack", "tier"]).reset_index(drop=True)

print(f"Feasible placements checked: {feasible_count}")
print(f"Best reshuffles (blocking pairs): {best_reshuffles}")
print(f"One best stack-choice sequence (0-based stacks): {best_seq}")

sol_df

In [ ]:
# ----------------------------
# Explain blocking relationships (which container blocks which?)
# ----------------------------
# This makes the reshuffle number easier to understand.
# We will list every pair:
# - lower container (departs earlier)
# - upper container (departs later)
# in the same stack.


def blocking_pairs(stack_config: List[List[str]]) -> List[Dict]:
    pairs: List[Dict] = []

    for s_idx, stack in enumerate(stack_config, start=1):
        for lower_idx in range(len(stack)):
            for upper_idx in range(lower_idx + 1, len(stack)):
                lower = stack[lower_idx]
                upper = stack[upper_idx]

                if dep_time[lower] < dep_time[upper]:
                    pairs.append(
                        {
                            "stack": s_idx,
                            "lower": lower,
                            "upper": upper,
                            "lower_departure": dep_time[lower],
                            "upper_departure": dep_time[upper],
                        }
                    )

    return pairs


bp = blocking_pairs(best_config)
blocking_df = pd.DataFrame(bp)

blocking_df

## Step 3 — Visualize the stacking plan and reshuffle logic

In yard operations, a picture is often the quickest way to understand a stacking rule.

Below we create three visuals:

- **Stack diagram** (which container is in which stack/tier)
- **Blocking arrows** (which containers block earlier departures)
- **Reshuffles by target container** (how many moves are needed to retrieve each container in departure order)

In [ ]:
# ----------------------------
# Visualization code
# ----------------------------
# We'll build a simple "stack chart".
# Each stack is a column.
# Each tier is a row (tier 0 at the bottom).
#
# Then we draw blocking arrows using `blocking_df`.

assert best_config is not None

# ----------------------------
# 1) Stack diagram
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 4))

# Visual settings
stack_width = 1.0
cell_height = 1.0
x_gap = 0.6

# Compute x positions for each stack
stack_x = {s_idx: (s_idx - 1) * (stack_width + x_gap) for s_idx in range(1, NUM_STACKS + 1)}

# Draw cells (empty slots up to MAX_TIERS)
for s_idx in range(1, NUM_STACKS + 1):
    x0 = stack_x[s_idx]
    for tier in range(MAX_TIERS):
        y0 = tier * cell_height
        rect = plt.Rectangle((x0, y0), stack_width, cell_height, facecolor="#F2F4F7", edgecolor="#344054")
        ax.add_patch(rect)

# Place containers as labels
for s_idx, stack in enumerate(best_config, start=1):
    x0 = stack_x[s_idx]
    for tier, cid in enumerate(stack):
        y0 = tier * cell_height
        ax.text(
            x0 + stack_width / 2,
            y0 + cell_height / 2,
            f"{cid}\n(d={dep_time[cid]})",
            ha="center",
            va="center",
            fontsize=10,
            color="#101828",
        )

# Title and axes formatting
ax.set_title("Optimal stacking plan (Tier 1 example)")
ax.set_xticks([stack_x[s] + stack_width / 2 for s in range(1, NUM_STACKS + 1)])
ax.set_xticklabels([f"Stack {s}" for s in range(1, NUM_STACKS + 1)])
ax.set_yticks([])

# Expand plot limits
ax.set_xlim(-0.5, stack_x[NUM_STACKS] + stack_width + 0.5)
ax.set_ylim(-0.2, MAX_TIERS * cell_height + 0.8)

# ----------------------------
# 2) Blocking arrows
# ----------------------------
# We draw an arrow from the "upper" container to the "lower" container.
# That visually means: "upper blocks lower".

if blocking_df is not None and len(blocking_df) > 0:
    for _, row in blocking_df.iterrows():
        s_idx = int(row["stack"])
        lower = row["lower"]
        upper = row["upper"]

        # Find tiers of lower and upper in this stack
        stack = best_config[s_idx - 1]
        lower_tier = stack.index(lower)
        upper_tier = stack.index(upper)

        # Compute centers
        x = stack_x[s_idx] + stack_width / 2
        y_lower = lower_tier * cell_height + cell_height / 2
        y_upper = upper_tier * cell_height + cell_height / 2

        ax.annotate(
            "",
            xy=(x, y_lower),
            xytext=(x, y_upper),
            arrowprops=dict(arrowstyle="->", lw=2, color="#F97066"),
        )

# Show the stack diagram with arrows
plt.show()

# ----------------------------
# 3) Reshuffles by target container (bar chart)
# ----------------------------
# For each container, how many later-departing containers are above it?
# This equals the number of blocking pairs where it is the "lower" container.

block_counts = {cid: 0 for cid, _ in containers}

if blocking_df is not None and len(blocking_df) > 0:
    for _, r in blocking_df.iterrows():
        block_counts[r["lower"]] += 1

# Retrieve order: increasing departure time
retrieve_order = sorted([cid for cid, _ in containers], key=lambda c: dep_time[c])

reshuffles_per_container = [block_counts[cid] for cid in retrieve_order]

plt.figure(figsize=(7, 3.2))
plt.bar(retrieve_order, reshuffles_per_container, color="#2E90FA", edgecolor="#101828", alpha=0.85)
plt.title("Reshuffles needed per container (by departure order)")
plt.xlabel("Container (retrieval order by departure time)")
plt.ylabel("Reshuffles (count)")
plt.grid(True, axis="y", alpha=0.25)
plt.show()

# Print the retrieval order explicitly
pd.DataFrame(
    {
        "container": retrieve_order,
        "departure_time": [dep_time[c] for c in retrieve_order],
        "reshuffles_needed": reshuffles_per_container,
    }
)

In [ ]:
# ----------------------------
# Additional logistics-style visuals
# ----------------------------
# Even in this simplified Tier 1 model, it is useful to see:
# - cumulative reshuffles as we progress through the retrieval order
# - a simple timeline view of "work" (base retrieval + reshuffles)

# Cumulative reshuffles in retrieval order
cum_resh = np.cumsum(reshuffles_per_container)

plt.figure(figsize=(7, 3.2))
plt.plot(retrieve_order, cum_resh, marker="o", lw=2, color="#344054")
plt.title("Cumulative reshuffles over retrieval order")
plt.xlabel("Container (retrieval order)")
plt.ylabel("Cumulative reshuffles")
plt.grid(True, alpha=0.25)
plt.show()

# Gantt-like timeline:
# We don't model actual minutes in Tier 1, but we can still visualize
# the number of moves per retrieval as a simple workload proxy.
#
# Assumption for visualization:
# - retrieving one container costs 1 base move
# - each reshuffle costs 1 additional move
moves_per_container = [1 + r for r in reshuffles_per_container]
starts = np.cumsum([0] + moves_per_container[:-1])

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(
    y=np.arange(len(retrieve_order)),
    width=moves_per_container,
    left=starts,
    color="#2E90FA",
    edgecolor="#101828",
    alpha=0.85,
)
ax.set_yticks(np.arange(len(retrieve_order)))
ax.set_yticklabels(retrieve_order)
ax.set_title("Timeline (Gantt-like) — workload proxy per retrieval")
ax.set_xlabel("Moves (1 retrieval + reshuffles)")
ax.set_ylabel("Container (retrieval order)")
ax.grid(True, axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## Common pitfalls and next steps

### Common pitfalls

- If reshuffles look too large or too small:
  - Double-check the reshuffle definition (blocking pairs) you are using.
  - Print the `blocking_df` table and confirm it matches your intuition from the stack diagram.
- If you want a more realistic reshuffle count:
  - You would need to simulate **actual retrieval operations** (moving blockers to temporary locations and possibly creating new blockings).

### Next steps

- Increase the number of containers and tiers and observe how exhaustive search becomes infeasible.
- In Tier 2, we replace exhaustive search with a fast rule (heuristic scoring).
- Try changing departure times and see how the optimal layout changes.